In [1]:
%run svm_eve.py

[INFO] Loading data...
[INFO] Extracting features... input shape: (1779, 1000)
[INFO] Features shape: (1779, 10)
[INFO] Saved test set (features) -> results/test_set_features_linear.npz
[INFO] Saved test set (features + raw) -> results/test_set_both_linear.npz
[INFO] Starting training with GridSearchCV...
Fitting 5 folds for each of 100 candidates, totalling 500 fits
[INFO] Training finished, time: 8.04 s
[INFO] Best parameters: {'svm__C': np.float64(0.13219411484660287), 'svm__kernel': 'linear'}
[INFO] Model saved to ./svm_model_linear.pkl
[INFO] Predicting on test set...
[INFO] Accuracy: 0.5787
[INFO] Classification report:
               precision    recall  f1-score   support

           0       0.61      0.44      0.51       177
           1       0.56      0.72      0.63       179

    accuracy                           0.58       356
   macro avg       0.58      0.58      0.57       356
weighted avg       0.58      0.58      0.57       356

[INFO] Macro F1: 0.5695
[INFO] Saved l

In [2]:
import joblib
import numpy as np

# load saved pipeline
model = joblib.load("svm_model_linear.pkl")

In [12]:
d = np.load("offts3_by_ij.npz")

rows = [np.asarray(d[f"{i}_{j}"]) for i in range(1,8) for j in range(1,4)]
min_len = min(len(r) for r in rows)

M = np.vstack([r[:min_len] for r in rows])  # (21, min_len)

In [13]:
def extract_features(data):
    """Extract statistical features from time distributions.

    Supports:
    - 1D input:  (n_features,)
    - 2D input:  (n_samples, n_features)

    Returns:
    - If input is 1D: shape (10,)
    - If input is 2D: shape (n_samples, 10)
    """
    data = np.asarray(data)

    if data.ndim == 1:
        data = data[None, :]   # convert to shape (1, n_timepoints)
        squeeze_output = True
    elif data.ndim == 2:
        squeeze_output = False
    else:
        raise ValueError("Input data must be a 1D or 2D array.")

    n_features = data.shape[1]

    mean = np.mean(data, axis=1, keepdims=True)
    std = np.std(data, axis=1, keepdims=True)
    cv = std / (mean + 1e-10)  # avoid division by zero
    max_ = np.max(data, axis=1, keepdims=True)

    percentile25 = np.percentile(data, 25, axis=1, keepdims=True)
    percentile50 = np.percentile(data, 50, axis=1, keepdims=True)
    percentile75 = np.percentile(data, 75, axis=1, keepdims=True)

    prop_gt1 = np.sum(data > 1, axis=1, keepdims=True) / n_features
    prop_gt2 = np.sum(data > 2, axis=1, keepdims=True) / n_features
    prop_gt3 = np.sum(data > 3, axis=1, keepdims=True) / n_features

    features = np.hstack([
        mean, std, cv, max_,
        prop_gt1, prop_gt2, prop_gt3,
        percentile25, percentile50, percentile75
    ])

    if squeeze_output:
        return features[0]   # return shape (10,) for 1D input
    return features

In [15]:
y_preds = []
y_pred_probas = []

for i in range(21):   # row 0 to row 20
    fmat = extract_features(rows[i]).reshape(1, -1)
    y_pred = model.predict(fmat)
    y_pred_proba = model.predict_proba(fmat)

    y_preds.append(y_pred[0])               # scalar label
    y_pred_probas.append(y_pred_proba[0])   # 1D probability vector

y_preds = np.array(y_preds)
y_pred_probas = np.array(y_pred_probas)

y_pred_probas

array([[0.55068377, 0.44931623],
       [0.42751337, 0.57248663],
       [0.43078871, 0.56921129],
       [0.62739307, 0.37260693],
       [0.39163513, 0.60836487],
       [0.41691563, 0.58308437],
       [0.70844044, 0.29155956],
       [0.44500543, 0.55499457],
       [0.41050118, 0.58949882],
       [0.66977745, 0.33022255],
       [0.38642145, 0.61357855],
       [0.38162353, 0.61837647],
       [0.71544635, 0.28455365],
       [0.48189813, 0.51810187],
       [0.42093948, 0.57906052],
       [0.64168394, 0.35831606],
       [0.46605086, 0.53394914],
       [0.42201118, 0.57798882],
       [0.75417041, 0.24582959],
       [0.41540509, 0.58459491],
       [0.41269415, 0.58730585]])